# Ch 21. RLHF — 解答

> このノートブックは 5 問すべての再現可能な解答を提供する。出力は消去済みで、コードセルは CPU のみの環境で上から順に実行できる。


## 問題 1

**問題**: REINFORCE で CartPole のような toy 環境を学習せよ。

### 解答

外部環境依存を避け、2 行動 bandit を toy 方策に使う。REINFORCE は $-(R-b)\log\pi(a)$ を最小化し、baseline は期待値を変えず分散を減らす。固定シード実行で良い行動の確率上昇を確認する。


In [ ]:
import numpy as np

rng=np.random.default_rng(2101); logits=np.zeros(2); rewards=np.array([0.2,0.8]); history=[]
for step in range(800):
    probs=np.exp(logits-logits.max()); probs/=probs.sum(); action=rng.choice(2,p=probs)
    reward=rewards[action]+rng.normal(scale=.05); advantage=reward-probs@rewards
    grad=-probs; grad[action]+=1; logits += .08*advantage*grad
    if step%100==0: history.append(float(probs[1]))
final=np.exp(logits-logits.max()); final/=final.sum()
assert final[1] > .9 and history[-1] > history[0]
print({"probability_better_action":round(float(final[1]),4),"learning_curve":np.round(history,3).tolist()})


## 問題 2

**問題**: PPO で $\epsilon=0.1,0.2,0.3$ を比較し、影響を分析せよ。

### 解答

$r=\pi_\theta/\pi_{old}$ の clip 区間は $[1-\epsilon,1+\epsilon]$ である。小さい $\epsilon$ は更新を保守的にし、大きい値はより大きな変化と不安定性を許す。固定 $r,A$ で目的値を厳密比較する。


In [ ]:
import numpy as np

ratios=np.linspace(.6,1.4,401); advantages=np.ones_like(ratios); results={}
for epsilon in (.1,.2,.3):
    surrogate=np.minimum(ratios*advantages,np.clip(ratios,1-epsilon,1+epsilon)*advantages)
    clipped_fraction=float(np.mean(ratios>1+epsilon)); max_objective=float(surrogate.max())
    results[epsilon]={"clipped_fraction":clipped_fraction,"max_surrogate":max_objective}
assert results[.1]["clipped_fraction"] > results[.2]["clipped_fraction"] > results[.3]["clipped_fraction"]
print({str(k):{m:round(v,4) for m,v in r.items()} for k,r in results.items()})


## 問題 3

**問題**: RM 学習で chosen と rejected が似ている場合と異なる場合の loss を比較せよ。

### 解答

RM loss は $-\log\sigma(r_c-r_r)$ である。差が 0 なら $\log2$、chosen が大きく優勢なら 0 に近づき、順序が逆なら増える。絶対点ではなく差だけが識別される。


In [ ]:
import numpy as np

gaps=np.array([0.0,0.5,2.0,-0.5]); losses=np.logaddexp(0,-gaps)
assert losses[2] < losses[1] < losses[0] < losses[3]
print([{"chosen_minus_rejected":float(g),"preference_loss":round(float(l),5)} for g,l in zip(gaps,losses)])


## 問題 4

**問題**: KL ペナルティ $\beta=0,0.1,1.0$ で方策変化をシミュレーションせよ。

### 解答

$R_{total}=R_{RM}-\beta D_{KL}$ では $\beta$ が大きいほど参照方策から離れる利得を抑える。1 次元凹代理目的で最適変化が $1/\beta$ に比例し、$\beta=0$ では有限の制約最適点がないことを示す。


In [ ]:
import numpy as np

# Optimize reward*delta - beta*delta^2/2 on a bounded grid; delta proxies policy movement/KL.
delta=np.linspace(0,3,3001); results={}
for beta in (0.0,.1,1.0):
    objective=delta-beta*delta**2/2
    optimum=float(delta[np.argmax(objective)])
    results[beta]={"optimal_policy_shift":optimum,"kl_proxy":optimum**2/2}
assert results[1.0]["optimal_policy_shift"] < results[.1]["optimal_policy_shift"] <= results[0.0]["optimal_policy_shift"]
print({str(k):{m:round(v,3) for m,v in r.items()} for k,r in results.items()})


## 問題 5

**問題**: RLHF に対する DPO の利点をメモリの観点から説明せよ。

### 解答

PPO 型 RLHF は policy、reference、reward、value モデルと rollout 活性を要する。DPO は通常、学習 policy と固定 reference の対数確率だけを使い、reward/value モデルとオンライン rollout を除くため、モデル状態と活性メモリが減る。


In [ ]:
# Count simultaneously resident trainable/frozen model-sized states under explicit assumptions.
ppo={"policy":1,"reference":1,"reward":1,"value":1,"rollout_buffer":.25}
dpo={"policy":1,"reference":1,"preference_batch":.05}
ppo_total=sum(ppo.values()); dpo_total=sum(dpo.values())
assert dpo_total < ppo_total and "reward" not in dpo and "value" not in dpo
print({"units_of_model_memory":{"PPO":ppo_total,"DPO":dpo_total},"relative_reduction":round(1-dpo_total/ppo_total,3),
       "assumption":"same precision; optimizer/activation memory excluded"})
